# PPMS（R–T、Tc）

和軟體用的是**同一份分析程式**（`magic/`），結果完全一樣；這裡適合自由探索、試新的分析。

- 第一格把 `DATA` 改成你的資料夾（沒填就用合成的範例資料）
- `run(機台, 分析名稱, [檔案], 參數=值…)` 回傳 `(fig, res)`：`res.text` 文字結果、`res.tables` 表格（pandas）
- 參數名稱和軟體參數面板一樣，可以用 `list_analyses("PPMS")` 看有哪些分析
- **上傳 GitHub 前會自動清掉輸出**（publish.bat），圖和數值不會被公開


In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # 沒有 pip install 也能 import magic（notebooks 的上一層就是工具資料夾）
from magic.runner import run, list_analyses
from magic.readers import load
from magic.core.plotstyle import apply
apply()                                          # 中文字型、配色和軟體一樣

DATA = r""   # ← PPMS 資料夾，例如 r"D:\Yeep\01 Magic Tool\PPMS\Data\20260615"
if not DATA:                                     # 沒填就用範例（合成資料）
    from magic.core.settings import data_dir
    from magic.testdata import write_all
    write_all(os.path.join(str(data_dir()), "demo"))
    DATA = os.path.join(str(data_dir()), "demo", "PPMS")
print("資料夾：", DATA)
print("可用的分析：", list_analyses("PPMS"))
sorted(os.listdir(DATA))

## 曲線檢視（任選兩欄，降溫 / 升溫自動分開）

In [ ]:
fp = os.path.join(DATA, sorted(f for f in os.listdir(DATA) if f.lower().endswith(".dat"))[0])   # ← 換成你的檔名
fig, res = run("PPMS", "曲線檢視", [fp], x="Temperature (K)", y="Bridge 1 Resistance (Ohms)")
fig

## R–T 與 Tc

`T_range` 空白 = 自動（最低溫到 2 × 轉變溫度）；`method` 可選 `50%（中點）`、`90%（onset）`、`10%（零電阻 / 完全抗磁）`、`最大斜率 |dy/dT|`。

In [ ]:
fig, res = run("PPMS", "R–T 與 Tc", [fp], T_range="2, 12", method="50%（中點）")
print(res.text)
res.tables["Tc"]

## 自由探索：直接拿 DataFrame

In [ ]:
ds = load(fp)
print(ds.kind, ds.header.get("BYAPP"), ds.notes)
ds.df.head()

In [ ]:
# 例：只取 4–10 K 的降溫段，存成 CSV（取代原本的 cell）
from magic.core.segments import split_sweeps
T = ds.df["Temperature (K)"].to_numpy()
R = ds.df["Bridge 1 Resistance (Ohms)"].to_numpy()
cool = [s for s in split_sweeps(T) if s.direction < 0][0]
import pandas as pd
part = pd.DataFrame({"T (K)": T[cool.slice], "R (Ohm)": R[cool.slice]})
part = part[(part["T (K)"] >= 4) & (part["T (K)"] <= 10)].sort_values("T (K)")
# part.to_csv(r"...\輸出.csv", index=False)      # 存到原始資料夾以外的地方
part.describe()